In [20]:
import pandas as pd
from arrlenhandling import lenhandling
import os
import re

class excelcleaner:
    @staticmethod
    def filecleaner(dataframe, newfilename: str):
        df = dataframe
        df = df.dropna(how='all')
        print(df.info())
        df.to_excel(newfilename, index=False, engine='openpyxl')

class FileReader:
    def __init__(self):
        self.keys_ = [
            "Job Role", "CTC", "Stipend", "Eligible Batch", 
            "Eligible Courses", "Eligible Branches", 
            "Internship Duration", "Location", "Application Link"
        ]
        
        self.company_dict = {
            "name": [],
            "Job Role": [],
            "CTC": [],
            "Stipend": [],
            "Eligible Batch": [],
            "Eligible Courses": [],
            "Eligible Branches": [],
            "Internship Duration": [],
            "Location": [],
            "Application Link": []
        }

    def extract_company_name(self, header_line):
        """Extract company name from header line"""
        # Remove date/phone prefix
        clean_line = re.sub(r'^\d{2}/\d{2}/\d{2},\s+\d{2}:\d{2}\s+-\s+\+[\d\s]+:\s+', '', header_line)
        clean_line = clean_line.strip('*').strip()
        
        # Get first part before |
        parts = clean_line.split('|')
        company_name = parts[0].strip() if parts else ""
        
        return company_name

    def normalize_key(self, line):
        """Check if line starts with any of our keys (including variations)"""
        line_lower = line.lower()
        
        # Direct matches
        for key in self.keys_:
            if line_lower.startswith(key.lower() + ":"):
                return key
        
        # Handle variations
        variations = {
            "eligible batches:": "Eligible Batch",
            "eligible course:": "Eligible Courses",
            "eligible branch:": "Eligible Branches",
            "application form:": "Application Link",
            "ctc (on ppo conversion):": "CTC"
        }
        
        for variation, normalized_key in variations.items():
            if line_lower.startswith(variation):
                return normalized_key
        
        return None

    def dictupdate(self, filename: str):
        with open(filename, "r", encoding="utf-8") as file:
            lines = file.readlines()
        
        i = 0
        company_count = 0
        
        while i < len(lines):
            line = lines[i].strip()
            
            # Skip empty lines
            if not line:
                i += 1
                continue
            
            # Check if this is a company header
            if line.startswith("*") and "|" in line:
                company_name = self.extract_company_name(line)
                
                # Skip if it's not a real company (like reminders, warnings, etc.)
                if not company_name or len(company_name) < 3:
                    i += 1
                    continue
                
                company_count += 1
                print(f"\n{'='*60}")
                print(f"COMPANY #{company_count}: {company_name}")
                print(f"{'='*60}")
                
                # Initialize temp dict
                temp_dict = {key: "" for key in self.company_dict.keys()}
                temp_dict["name"] = company_name
                
                # Process subsequent lines
                i += 1
                current_key = None
                current_value = []
                
                while i < len(lines):
                    line = lines[i].strip()
                    
                    # Check if we've reached next company
                    if line.startswith("*") and "|" in line:
                        next_company = self.extract_company_name(line)
                        if next_company and len(next_company) >= 3:
                            break
                    
                    if not line:
                        i += 1
                        continue
                    
                    # Check if this line is a key
                    matched_key = self.normalize_key(line)
                    
                    if matched_key:
                        # Save previous key-value
                        if current_key:
                            value = " | ".join([v for v in current_value if v])
                            temp_dict[current_key] = value
                            print(f"  {current_key}: {value}")
                        
                        # Start new key-value
                        current_key = matched_key
                        value_part = line.split(":", 1)[1].strip() if ":" in line else ""
                        current_value = [value_part] if value_part else []
                    elif current_key:
                        # Continuation of current key
                        clean_line = line.lstrip('*').lstrip('•').lstrip('-').lstrip('⁠').strip()
                        if clean_line and not clean_line.startswith("_"):  # Skip notes
                            current_value.append(clean_line)
                    
                    i += 1
                
                # Save last key-value
                if current_key:
                    value = " | ".join([v for v in current_value if v])
                    temp_dict[current_key] = value
                    print(f"  {current_key}: {value}")
                
                # Add to main dict
                for key in self.company_dict.keys():
                    self.company_dict[key].append(temp_dict[key])
                
                continue
            
            i += 1
        
        print(f"\n{'='*60}")
        print(f"TOTAL COMPANIES FOUND: {company_count}")
        print(f"{'='*60}\n")
        
        # Length handling
        handler = lenhandling()
        self.company_dict = handler.lenhandling(self.company_dict)

        try:
            df = pd.DataFrame.from_dict(self.company_dict)
            print("\nDataFrame Preview:")
            print(df.to_string())
            
            newfilename = input("\nEnter the filename to save the excel (without .xlsx extension): ")
            newfilename_path = os.path.join(os.getcwd(), newfilename + ".xlsx")
            print(f"Saving to: {newfilename_path}")
            
            excel_cleaner = excelcleaner()
            excel_cleaner.filecleaner(df, newfilename_path)
        except Exception as e:
            print("Error writing Excel:", e)
            raise
        finally:
            print("Dictionary update completed.")

if __name__ == "__main__":
    try:
        file_reader = FileReader()
        file_reader.dictupdate(r"D:\chatexport\chats.txt")
    except Exception as e:
        print("Error:", e)
        import traceback
        traceback.print_exc()
    finally:
        print("Execution completed.")


TOTAL COMPANIES FOUND: 0


DataFrame Preview:
Empty DataFrame
Columns: [name, Job Role, CTC, Stipend, Eligible Batch, Eligible Courses, Eligible Branches, Internship Duration, Location, Application Link]
Index: []
Saving to: d:\chatexport\one.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   name                 0 non-null      float64
 1   Job Role             0 non-null      float64
 2   CTC                  0 non-null      float64
 3   Stipend              0 non-null      float64
 4   Eligible Batch       0 non-null      float64
 5   Eligible Courses     0 non-null      float64
 6   Eligible Branches    0 non-null      float64
 7   Internship Duration  0 non-null      float64
 8   Location             0 non-null      float64
 9   Application Link     0 non-null      float64
dtypes: float64(10)
memory usage: 124.0 bytes
None
Diction

In [21]:
import re

def debug_file_reader(filename: str):
    """Debug script to see what's in the file"""
    with open(filename, "r", encoding="utf-8") as file:
        lines = file.readlines()
    
    print(f"Total lines in file: {len(lines)}")
    print(f"\n{'='*60}")
    print("First 50 lines:")
    print(f"{'='*60}\n")
    
    for i, line in enumerate(lines[:50]):
        # Show line number, repr (to see hidden chars), and the line itself
        print(f"Line {i}: {repr(line)}")
    
    print(f"\n{'='*60}")
    print("Lines containing '*' and '|':")
    print(f"{'='*60}\n")
    
    for i, line in enumerate(lines):
        if '*' in line and '|' in line:
            print(f"Line {i}: {line.strip()}")
            print(f"  Raw: {repr(line)}")
            
            # Try to extract company name
            clean_line = re.sub(r'^\d{2}/\d{2}/\d{2},\s+\d{2}:\d{2}\s+-\s+\+[\d\s]+:\s+', '', line)
            clean_line = clean_line.strip('*').strip()
            parts = clean_line.split('|')
            company_name = parts[0].strip() if parts else ""
            print(f"  Extracted company: '{company_name}'")
            print()

if __name__ == "__main__":
    debug_file_reader(r"D:\chatexport\chats.txt")

Total lines in file: 102

First 50 lines:

Line 0: '08/09/25, 17:11 - +91 97773 92266: *AB InBev | Summer Internship | On-Campus*\n'
Line 1: '\n'
Line 2: '*Job Role:* Supply Excellence Trainee Intern\n'
Line 3: '\n'
Line 4: '*Stipend:* INR 60 KPM + INR 15000 (Reimbursement against rent, utilities)\n'
Line 5: '\n'
Line 6: '*CTC (On PPO Conversion):* 17.5 LPA\n'
Line 7: '\n'
Line 8: '*Eligible Batch:* 2027\n'
Line 9: '\n'
Line 10: '*Eligible Courses:* B.Tech\n'
Line 11: '\n'
Line 12: '*Eligible Branches:*\n'
Line 13: '* EC, EI, EE, ME, CH, CE, MM, FP\n'
Line 14: ' \n'
Line 15: '*Eligibility:*\n'
Line 16: '* \u2060No active backlogs\n'
Line 17: '\n'
Line 18: '*Selection Process:*\n'
Line 19: '* Pre-Placement Talk\n'
Line 20: '* Resume Shortlisting\n'
Line 21: '* Group Discussion\n'
Line 22: '* Personal Interview\n'
Line 23: '\n'
Line 24: '\n'
Line 25: '*Application Form:* https://forms.gle/2mawSG2Sy8h1h95R8\n'
Line 26: '\n'
Line 27: '*Deadline:* 11.59PM, 9th September, 2025\n'
Line 28: '\

In [ ]:
import pandas as pd
from arrlenhandling import lenhandling
import os
import re

class excelcleaner:
    @staticmethod
    def filecleaner(dataframe, newfilename: str):
        df = dataframe
        df = df.dropna(how='all')
        print(df.info())
        df.to_excel(newfilename, index=False, engine='openpyxl')

class FileReader:
    def __init__(self):
        self.keys_ = [
            "Job Role", "CTC", "Stipend", "Eligible Batch", 
            "Eligible Courses", "Eligible Branches", 
            "Internship Duration", "Location", "Application Link"
        ]
        
        self.company_dict = {
            "name": [],
            "Job Role": [],
            "CTC": [],
            "Stipend": [],
            "Eligible Batch": [],
            "Eligible Courses": [],
            "Eligible Branches": [],
            "Internship Duration": [],
            "Location": [],
            "Application Link": []
        }

    def extract_company_name(self, header_line):
        """Extract company name from header line"""
        # Remove date/phone prefix
        clean_line = re.sub(r'^\d{2}/\d{2}/\d{2},\s+\d{2}:\d{2}\s+-\s+\+[\d\s]+:\s+', '', header_line)
        clean_line = clean_line.strip('*').strip()
        
        # Get first part before |
        parts = clean_line.split('|')
        company_name = parts[0].strip() if parts else ""
        
        return company_name

    def normalize_key(self, line):
        """Check if line starts with any of our keys (including variations)"""
        # Remove leading/trailing asterisks and whitespace
        clean_line = line.strip('*').strip()
        line_lower = clean_line.lower()
        
        # Direct matches
        for key in self.keys_:
            if line_lower.startswith(key.lower() + ":"):
                return key
        
        # Handle variations
        variations = {
            "eligible batches:": "Eligible Batch",
            "eligible course:": "Eligible Courses",
            "eligible branch:": "Eligible Branches",
            "application form:": "Application Link",
            "ctc (on ppo conversion):": "CTC"
        }
        
        for variation, normalized_key in variations.items():
            if line_lower.startswith(variation):
                return normalized_key
        
        return None

    def dictupdate(self, filename: str):
        with open(filename, "r", encoding="utf-8") as file:
            lines = file.readlines()
        
        i = 0
        company_count = 0
        
        while i < len(lines):
            line = lines[i].strip()
            
            # Skip empty lines
            if not line:
                i += 1
                continue
            
            # Check if this is a company header (has date/phone prefix and contains | )
            if re.match(r'^\d{2}/\d{2}/\d{2},\s+\d{2}:\d{2}\s+-\s+\+[\d\s]+:', line) and '|' in line:
                company_name = self.extract_company_name(line)
                
                # Skip if company name is too short or empty
                if not company_name or len(company_name) < 2:
                    i += 1
                    continue
                
                company_count += 1
                print(f"\n{'='*60}")
                print(f"COMPANY #{company_count}: {company_name}")
                print(f"{'='*60}")
                
                # Initialize temp dict
                temp_dict = {key: "" for key in self.company_dict.keys()}
                temp_dict["name"] = company_name
                
                # Process subsequent lines
                i += 1
                current_key = None
                current_value = []
                
                while i < len(lines):
                    line = lines[i].strip()
                    
                    # Check if we've reached next company (next message with date/phone)
                    if re.match(r'^\d{2}/\d{2}/\d{2},\s+\d{2}:\d{2}\s+-\s+\+[\d\s]+:', line):
                        break
                    
                    if not line:
                        i += 1
                        continue
                    
                    # Check if this line is a key (format: *Key:* value or *Key:*)
                    matched_key = self.normalize_key(line)
                    
                    if matched_key:
                        # Save previous key-value
                        if current_key:
                            value = " | ".join([v for v in current_value if v])
                            temp_dict[current_key] = value
                            print(f"  {current_key}: {value}")
                        
                        # Start new key-value
                        current_key = matched_key
                        # Extract value after the colon (remove asterisks)
                        value_part = line.strip('*').split(":", 1)[1].strip() if ":" in line else ""
                        current_value = [value_part] if value_part else []
                    elif current_key:
                        # Continuation of current key
                        # Remove leading asterisks, bullets, and special chars
                        clean_line = line.lstrip('*').lstrip('•').lstrip('-').lstrip('\u2060').strip()
                        # Skip lines that are clearly notes or other sections we don't want
                        if clean_line and not clean_line.startswith("_"):
                            current_value.append(clean_line)
                    
                    i += 1
                
                # Save last key-value
                if current_key:
                    value = " | ".join([v for v in current_value if v])
                    temp_dict[current_key] = value
                    print(f"  {current_key}: {value}")
                
                # Add to main dict
                for key in self.company_dict.keys():
                    self.company_dict[key].append(temp_dict[key])
                
                continue
            
            i += 1
        
        print(f"\n{'='*60}")
        print(f"TOTAL COMPANIES FOUND: {company_count}")
        print(f"{'='*60}\n")
        
        # Length handling
        handler = lenhandling()
        self.company_dict = handler.lenhandling(self.company_dict)

        try:
            df = pd.DataFrame.from_dict(self.company_dict)
            print("\nDataFrame Preview:")
            print(df.to_string())
            
            newfilename = input("\nEnter the filename to save the excel (without .xlsx extension): ")
            newfilename_path = os.path.join(os.getcwd(), newfilename + ".xlsx")
            print(f"Saving to: {newfilename_path}")
            
            excel_cleaner = excelcleaner()
            excel_cleaner.filecleaner(df, newfilename_path)
        except Exception as e:
            print("Error writing Excel:", e)
            raise
        finally:
            print("Dictionary update completed.")

if __name__ == "__main__":
    try:
        file_reader = FileReader()
        chat_text=input("Enter chat text file path: ")
        file_reader.dictupdate(chat_text)
    except Exception as e:
        print("Error:", e)
        import traceback
        traceback.print_exc()
    finally:
        print("Execution completed.")


COMPANY #1: AB InBev
  Job Role: * Supply Excellence Trainee Intern
  Stipend: * INR 60 KPM + INR 15000 (Reimbursement against rent, utilities)
  CTC: * 17.5 LPA
  Eligible Batch: * 2027
  Eligible Courses: * B.Tech
  Eligible Branches: EC, EI, EE, ME, CH, CE, MM, FP | Eligibility:* | ⁠No active backlogs | Selection Process:* | Pre-Placement Talk | Resume Shortlisting | Group Discussion | Personal Interview
  Application Link: * https://forms.gle/2mawSG2Sy8h1h95R8 | Deadline:* 11.59PM, 9th September, 2025 | Note:* | Keep the short Deadline in mind.

COMPANY #2: Jindal Steel & Power Ltd.

COMPANY #3: JSPL

COMPANY #4: Jindal Steel and Power Ltd. (JSPL)

COMPANY #5: Jindal Steel and Power Ltd. (JSPL)

COMPANY #6: Cisco

COMPANY #7: Aditya Birla Group
  Eligible Batch: * 2026, 2027
  Eligible Courses: * B.Tech
  Eligible Branches: * EC, EI, EE, ME, CH, CE, MM, MN, CR | Venue:* TIIR Auditorium | Date:* 15 September 2025 (Tomorrow) | Time:* 14:00 IST (2:00 PM)

TOTAL COMPANIES FOUND: 7


D

In [6]:
import pandas as pd
df=pd.read_csv(r"D:\chatexport\2025-10-26T13-43_export.csv")
print(len(df["name"].unique()))


328
